# 01 — Autoencoder Fundamentals

**Goal:** Build a simple dense autoencoder on Fashion-MNIST, train it to reconstruct images through a 32-dim latent bottleneck, and visualize the learned latent space.

**What you'll see:**
- Encoder–bottleneck–decoder architecture in Keras
- Reconstruction loss curve
- Side-by-side originals vs. reconstructions
- 2D PCA projection of the 32-dim latent space, colored by class

**Author:** Zain Rafeeque

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.decomposition import PCA

tf.random.set_seed(42)
np.random.seed(42)
print('TensorFlow', tf.__version__)

## 1. Load Fashion-MNIST

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0
x_train_flat = x_train.reshape(-1, 784)
x_test_flat  = x_test.reshape(-1, 784)

CLASS_NAMES = ['T-shirt','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Boot']
print('train', x_train_flat.shape, 'test', x_test_flat.shape)

## 2. Build the autoencoder

Compress 784-dim images → **32-dim latent** → reconstruct 784-dim. That's a 24× compression ratio; the model has to learn what features matter most.

In [ ]:
LATENT_DIM = 32

def build_autoencoder(latent_dim):
    encoder = models.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(256, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(latent_dim, activation='relu', name='latent'),
    ], name='encoder')

    decoder = models.Sequential([
        layers.Input(shape=(latent_dim,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(256, activation='relu'),
        layers.Dense(784, activation='sigmoid'),
    ], name='decoder')

    autoencoder = models.Sequential([encoder, decoder], name='autoencoder')
    return autoencoder, encoder, decoder

autoencoder, encoder, decoder = build_autoencoder(LATENT_DIM)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

## 3. Train

In [ ]:
history = autoencoder.fit(
    x_train_flat, x_train_flat,
    epochs=15,
    batch_size=256,
    validation_data=(x_test_flat, x_test_flat),
    callbacks=[callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=2,
)

final_val = history.history['val_loss'][-1]
print(f'Final validation MSE: {final_val:.4f}')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Reconstruction loss (MSE)')
plt.xlabel('epoch'); plt.ylabel('MSE'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Visual reconstruction quality

In [ ]:
n = 8
idx = np.random.choice(len(x_test_flat), n, replace=False)
originals = x_test_flat[idx]
reconstructions = autoencoder.predict(originals, verbose=0)

fig, axes = plt.subplots(2, n, figsize=(n * 1.6, 3.5))
for i in range(n):
    axes[0, i].imshow(originals[i].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(reconstructions[i].reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_title('Original', loc='left')
axes[1, 0].set_title('Reconstructed', loc='left')
plt.tight_layout(); plt.show()

## 5. Latent space visualization

Encode the test set into 32-dim, then PCA down to 2-dim and color by class. If the autoencoder learned meaningful structure, classes will form distinguishable clusters even though training was unsupervised.

In [ ]:
z_test = encoder.predict(x_test_flat, batch_size=512, verbose=0)
z2 = PCA(n_components=2, random_state=42).fit_transform(z_test)

plt.figure(figsize=(8, 7))
for cls in range(10):
    mask = y_test == cls
    plt.scatter(z2[mask, 0], z2[mask, 1], s=4, alpha=0.5, label=CLASS_NAMES[cls])
plt.legend(markerscale=2, loc='upper right', fontsize=8)
plt.title(f'Latent space (PCA 2D from {LATENT_DIM}-dim) — Fashion-MNIST')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Takeaways

- A 24× compression bottleneck still produces visually coherent reconstructions.
- The unsupervised latent space organizes Fashion-MNIST into class clusters — footwear (sneaker / sandal / boot) groups separately from upper-body garments (T-shirt / pullover / shirt / coat).
- Final validation MSE printed above is the baseline we'll improve in the next notebook with a convolutional architecture.